# Step 4: Analyze Results & Generate Plots

**CMPE 260 — Group 6**

This notebook reads the CSVs from Steps 1-3 and generates:
- Training curve comparison plots
- Degradation curves (shift level vs score)
- Robustness heatmap
- AUDC comparison bar chart
- Summary tables for the report

**No GPU needed** — runs on CPU or locally.

**Prerequisites:** Upload CSVs from Steps 1-3 to `results/` folder.

In [ ]:
import csv, glob, os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

ENVIRONMENTS = ['hopper-medium-v2', 'halfcheetah-medium-v2', 'walker2d-medium-v2']
GRAVITY_LEVELS = [0.5, 1.0, 1.5, 2.0]
OBS_NOISE_LEVELS = [0.0, 0.01, 0.1, 0.3]

def load_csv(filepath):
    rows = []
    with open(filepath) as f:
        for row in csv.DictReader(f):
            for k in row:
                try: row[k] = float(row[k])
                except: pass
            rows.append(row)
    return rows

def compute_audc(results, shift_type):
    filtered = sorted([r for r in results if r['shift_type'] == shift_type],
                      key=lambda x: x['shift_level'])
    if len(filtered) < 2: return 0.0
    return np.trapz([abs(r['robustness_drop']) for r in filtered],
                    [r['shift_level'] for r in filtered])

print('Ready. Upload CSVs to results/ folder.')

In [ ]:
# Upload CSVs if running on Colab
# from google.colab import files
# uploaded = files.upload()  # Upload all CSVs from Steps 1-3
# os.makedirs('results', exist_ok=True)
# for name, data in uploaded.items():
#     with open(f'results/{name}', 'wb') as f: f.write(data)

# List available results
print('Available result files:')
for f in sorted(glob.glob('results/*.csv')):
    print(f'  {f}')

## Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, env_name in enumerate(ENVIRONMENTS):
    ax = axes[idx]
    for nq, color, style in [(2, 'steelblue', '-o'), (3, 'coral', '--s')]:
        fpath = f'results/training_{env_name}_{nq}Q.csv'
        if os.path.exists(fpath):
            data = load_csv(fpath)
            ax.plot([r['step'] for r in data], [r['normalized_score'] for r in data],
                    style, label=f'{nq}Q', color=color, linewidth=2, markersize=6)
    ax.set_xlabel('Training Steps')
    ax.set_ylabel('Normalized Score')
    ax.set_title(env_name)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Training Curves — Baseline (2Q) vs Q-Ensemble (3Q)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/plot_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## Degradation Curves

In [ ]:
fig, axes = plt.subplots(len(ENVIRONMENTS), 2, figsize=(14, 5*len(ENVIRONMENTS)))

for row, env_name in enumerate(ENVIRONMENTS):
    for col, (shift_type, levels) in enumerate([('gravity', GRAVITY_LEVELS), ('obs_noise', OBS_NOISE_LEVELS)]):
        ax = axes[row, col] if len(ENVIRONMENTS) > 1 else axes[col]
        for nq, color, style in [(2, 'steelblue', '-o'), (3, 'coral', '--s')]:
            fpath = f'results/shift_{env_name}_{nq}Q.csv'
            if os.path.exists(fpath):
                data = load_csv(fpath)
                filtered = sorted([r for r in data if r['shift_type'] == shift_type],
                                  key=lambda x: x['shift_level'])
                ax.plot([r['shift_level'] for r in filtered],
                        [r['raw_return'] for r in filtered],
                        style, label=f'{nq}Q', color=color, linewidth=2, markersize=8)
        ax.set_xlabel('Gravity Scale' if shift_type == 'gravity' else 'Noise σ')
        ax.set_ylabel('Normalized Score')
        ax.set_title(f'{env_name} — {shift_type}')
        ax.legend()
        ax.grid(True, alpha=0.3)

plt.suptitle('Performance Under Distribution Shift', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('results/plot_degradation_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## Robustness Heatmap

In [ ]:
# Build heatmap: rows = (env, critic), cols = shift conditions
shift_labels = [f'G={l}' for l in GRAVITY_LEVELS] + [f'N={l}' for l in OBS_NOISE_LEVELS]
row_labels = []
heatmap_data = []

for env_name in ENVIRONMENTS:
    short = env_name.split('-')[0].capitalize()
    for nq in [2, 3]:
        fpath = f'results/shift_{env_name}_{nq}Q.csv'
        if os.path.exists(fpath):
            data = load_csv(fpath)
            row = []
            for st, levels in [('gravity', GRAVITY_LEVELS), ('obs_noise', OBS_NOISE_LEVELS)]:
                for level in levels:
                    match = [r for r in data if r['shift_type'] == st and r['shift_level'] == level]
                    row.append(match[0]['robustness_drop'] if match else float('nan'))
            heatmap_data.append(row)
            row_labels.append(f'{short} {nq}Q')

if heatmap_data:
    fig, ax = plt.subplots(figsize=(14, max(4, len(row_labels) * 0.6)))
    data_arr = np.array(heatmap_data)
    im = ax.imshow(data_arr, cmap='RdYlGn_r', aspect='auto', vmin=-0.5, vmax=1.0)
    ax.set_xticks(range(len(shift_labels)))
    ax.set_xticklabels(shift_labels, rotation=45, ha='right')
    ax.set_yticks(range(len(row_labels)))
    ax.set_yticklabels(row_labels)
    for i in range(len(row_labels)):
        for j in range(len(shift_labels)):
            val = data_arr[i, j]
            if not np.isnan(val):
                ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                        color='white' if abs(val) > 0.3 else 'black', fontsize=9)
    plt.colorbar(im, ax=ax, label='Δ(δ) Robustness Drop')
    ax.set_title('Robustness Drop Heatmap — All Environments & Shift Conditions')
    plt.tight_layout()
    plt.savefig('results/plot_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No shift results found. Run Step 3 first.')

## AUDC Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for col, shift_type in enumerate(['gravity', 'obs_noise']):
    ax = axes[col]
    env_labels = []
    audc_2q = []
    audc_3q = []
    for env_name in ENVIRONMENTS:
        short = env_name.split('-')[0].capitalize()
        env_labels.append(short)
        for nq, lst in [(2, audc_2q), (3, audc_3q)]:
            fpath = f'results/shift_{env_name}_{nq}Q.csv'
            if os.path.exists(fpath):
                lst.append(compute_audc(load_csv(fpath), shift_type))
            else:
                lst.append(0)

    x = np.arange(len(env_labels))
    w = 0.35
    ax.bar(x - w/2, audc_2q, w, label='Baseline (2Q)', color='steelblue')
    ax.bar(x + w/2, audc_3q, w, label='Q-Ensemble (3Q)', color='coral')
    ax.set_xticks(x)
    ax.set_xticklabels(env_labels)
    ax.set_ylabel('AUDC (lower = more robust)')
    ax.set_title(f'{shift_type.replace("_", " ").title()} — AUDC')
    ax.legend()

plt.suptitle('Area Under Degradation Curve', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/plot_audc.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary Table (for the report)

In [ ]:
print(f"\n{'='*90}")
print(f"FULL RESULTS TABLE")
print(f"{'='*90}")
print(f"{'Environment':<25} {'Shift':<12} {'Level':<8} {'2Q Score':<12} {'2Q Δ(δ)':<10} {'3Q Score':<12} {'3Q Δ(δ)':<10}")
print(f"{'-'*89}")

for env_name in ENVIRONMENTS:
    data_2q = load_csv(f'results/shift_{env_name}_2Q.csv') if os.path.exists(f'results/shift_{env_name}_2Q.csv') else []
    data_3q = load_csv(f'results/shift_{env_name}_3Q.csv') if os.path.exists(f'results/shift_{env_name}_3Q.csv') else []

    for st, levels in [('gravity', GRAVITY_LEVELS), ('obs_noise', OBS_NOISE_LEVELS)]:
        for level in levels:
            r2 = next((r for r in data_2q if r['shift_type'] == st and r['shift_level'] == level), None)
            r3 = next((r for r in data_3q if r['shift_type'] == st and r['shift_level'] == level), None)
            s2 = f"{r2['raw_return']:.2f}" if r2 else 'N/A'
            d2 = f"{r2['robustness_drop']:.4f}" if r2 else 'N/A'
            s3 = f"{r3['raw_return']:.2f}" if r3 else 'N/A'
            d3 = f"{r3['robustness_drop']:.4f}" if r3 else 'N/A'
            print(f"{env_name:<25} {st:<12} {level:<8.2f} {s2:<12} {d2:<10} {s3:<12} {d3:<10}")
    print()

In [ ]:
# Download all plots
# from google.colab import files
# for f in glob.glob('results/plot_*.png'):
#     files.download(f)

print('\nGenerated plots:')
for f in sorted(glob.glob('results/plot_*.png')):
    print(f'  {f}')